In [1]:
!pip install pybullet opencv-python matplotlib numpy

In [1]:
import numpy as np
import pybullet as p
import pybullet_data
import time
import random
import os


In [ ]:
p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
planeId = p.loadURDF("plane.urdf")

Loaded PyBullet plane.


In [ ]:
# Make the uv mapped sphere mesh for the textures to wrap around
def make_uv_sphere_obj(filename, stacks=32, slices=32):
    verts, uvs, faces = [], [], []
    for i in range(stacks + 1):
        v = i / stacks
        theta = v * np.pi
        for j in range(slices + 1):
            u = j / slices
            phi = u * 2 * np.pi
            verts.append((np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)))
            uvs.append((u, 1 - v))
    for i in range(stacks):
        for j in range(slices):
            a = i * (slices + 1) + j + 1
            b = a + slices + 1
            faces.append((a, a + 1, b + 1, b))
    with open(filename, "w") as f:
        for x, y, z in verts: f.write(f"v {x} {y} {z}\n")
        for u, w in uvs:      f.write(f"vt {u} {w}\n")
        for a, b, c, d in faces:
            f.write(f"f {a}/{a} {b}/{b} {c}/{c} {d}/{d}\n")

SPHERE_OBJ = os.path.join(os.getcwd(), "uv_sphere.obj")
make_uv_sphere_obj(SPHERE_OBJ)


# Creates tennis, ping=pong, and baseball 
def create_sports_ball(radius, mass, lateral_friction, rolling_friction,
                       color=None, texture_path=None, position=[0, 0, 0]):
    collision_shape = p.createCollisionShape(p.GEOM_SPHERE, radius=radius)

    if texture_path:
        visual_shape = p.createVisualShape(
            shapeType=p.GEOM_MESH, fileName=SPHERE_OBJ,
            meshScale=[radius]*3, rgbaColor=[1, 1, 1, 1]
        )
    else:
        if color is None: color = [1, 1, 1, 1]
        visual_shape = p.createVisualShape(
            shapeType=p.GEOM_SPHERE, radius=radius, rgbaColor=color
        )

    ball_id = p.createMultiBody(
        baseMass=mass,
        baseCollisionShapeIndex=collision_shape,
        baseVisualShapeIndex=visual_shape, basePosition=position
    )
    p.changeDynamics(ball_id, -1,
                     lateralFriction=lateral_friction,
                     rollingFriction=rolling_friction,
                     restitution=0.75)

    if texture_path:
        tex = p.loadTexture(texture_path)
        p.changeVisualShape(ball_id, -1, textureUniqueId=tex)

    return ball_id

# Real-world gravity
p.setGravity(0, 0, -9.81)

tennis_ball = create_sports_ball(
    radius=0.0327, mass=0.058,
    lateral_friction=0.6, rolling_friction=0.01,
    position=[random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), 0.5],
    texture_path=r"c:\SB VexPushback\EZ-Template-Example-Project (1)\.vscode\Screenshot 2026-06-24 201507.jpg"
)

baseball = create_sports_ball(
    radius=0.037, mass=0.145,
    lateral_friction=0.5, rolling_friction=0.005,
    position=[random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), 0.5],
    texture_path=r"c:\SB VexPushback\EZ-Template-Example-Project (1)\.vscode\Screenshot 2026-06-24 220302.jpg"
)

ping_pong = create_sports_ball(
    radius=0.02, mass=0.0027,
    lateral_friction=0.2, rolling_friction=0.001,
    position=[random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), 0.5],
    color=[0.9, 0.55, 0.1, 1]
)

for _ in range(2000):
    p.stepSimulation()
    time.sleep(1./240.)

In [ ]:

p.setGravity(0, 0, -9.81)

tray_scale = 0.3   # makes trays smaller than default

tray1 = p.loadURDF("tray/traybox.urdf", basePosition=[1,  0.0, 0], globalScaling=tray_scale)
tray2 = p.loadURDF("tray/traybox.urdf", basePosition=[1,  0.5, 0], globalScaling=tray_scale)
tray3 = p.loadURDF("tray/traybox.urdf", basePosition=[1, -0.5, 0], globalScaling=tray_scale)

Three trays loaded.
